In [30]:
import numpy as np
from scipy import linalg as la
import pandas as pd
import sympy as sp
np.set_printoptions(precision=2,suppress=True)

#####
<font color="#2c3e50" size="6"><div style="
    font-family: sans-serif;
    font-size: 1.2em;
    font-weight: 700;
    line-height: 1.2;
    background-color: #d9e6f2;
    border-radius: 8px;
    border-top: 2px solid rgba(144,62,80,0.3);
    border-bottom: 2px solid rgba(144,62,80,0.3);
    padding: 0.5em 1em;
    margin-top: 1em;
    margin-bottom: 0.6em;
    box-shadow: 0 2px 6px rgba(0,0,0,0.1);
" align="center">
1.8 Applications
</div></font>

######
<font color="#3f7fff" size="5"><div style="
    font-family: Verdana;
    font-size: 1.6em;
    font-weight: 600;
    line-height: 1.3;
    padding: 0.5em 1em;
    margin: 0.5em 0 0.6em 0;
    border-top: 2px solid rgba(144,62,80,0.3);
    border-bottom: 2px solid rgba(144,62,80,0.3);
    border-radius: 8px;
">
<b>Correlation and Cosine Similarity</b>
</div></font> 


In chapter 1.3, we looked at a correlation heatmap. As a reminder, a correlation coefficient is a single number that quantifies
the linear relationship between two variables. Correlation coefficients range from $−1$ to $+1$:
* $−1$ indicates a perfect negative relationship
* $+1$ indicates a perfect positive relationship
* $0$ indicates no linear correlation.

In data science, the variables (or features) can be viewed as vectors. Computing the correlation coefficient between two features can therefore be interpreted using linear algebra. To compute the correlation coefficient, we first perform the following steps:

1. Center the data by subtracting the mean from each data point. This produces feature vectors whose mean is zero.
2. Normalize the data by dividing each feature vector by its norm.

If you have taken a statistics course, you may already be familiar with the formula for **correlation coefficient** (also known as the **Pearson correlation coefficient**). 
$$\rho=\frac{\sum_{i=1}^n(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_{i=1}^n(x_i-\bar{x})^2}\sqrt{\sum_{i=1}^n(y_i-\bar{y})^2}}$$

This may look quite scary. However, from a linear algebra perspective it can be written much more compactly. Let $\tilde{\bf x}$ and $\tilde{\bf y}$ denote centered vectors of $\bf x$ and $\bf y$. Then the formula becomes 
$$\rho=\frac{\tilde{\bf x}\cdot \tilde{\bf y}}{||\tilde{\bf x}||\text{ }||\tilde{\bf y}||}$$

Notice that if the vectors are already centered and normalized, then the correlation coefficient simplifies to just the dot product of the vectors. This will be helpful in the future.

Another method to measure similarity between vectors is **cosine similarity**, which is simply the cosine of the angle between two vectors. Using the geometric definition of the dot product, we obtain

$$\cos(\theta)=\frac{{\bf x}\cdot {\bf y}}{||{\bf x}||\text{ }||{\bf y}||}$$

This formula looks very similar to the Pearson correlation coefficient. The key difference is that cosine similarity does not require the data to be centered.


<div style="
    font-family: Verdana, sans-serif;
    font-size: 14px;
    line-height: 1.6;
    margin: 0.8em 0;
    padding: 0.75em 1em;
    border-radius: 6px;
    border-left: 4px solid #2c3e50;
    border-left-color: #4c72b0;
    background-color: rgba(10,150,250,0.16);
    margin: 0.4em 0;
">
<strong>📌 Note:</strong> Cosine similarity is widely used in data mining and machine learning, including large language models such as ChatGPT. It is particularly useful for measuring similarity between high-dimensional vectors (for example, text embeddings).
</div>

---
**<font color="red">Example 1</font>**

Let's load the Penguin dataset and compute the correlation coefficient and cosine similarity between flipper length and body mass.

In [25]:
df=pd.read_csv('Penguins_map.csv')

In [26]:
x=df['flipper_length_mm'].to_numpy()
y=df['body_mass_g'].to_numpy()

Let's center these features:

In [27]:
xn=(x-x.mean())
yn=(y-y.mean())

Then correlation coefficient is 

In [28]:
corr=np.dot(xn,yn)/(la.norm(xn)*la.norm(yn))
print(f'Correlation coefficient = {corr:.3f}')

Correlation coefficient = 0.873


Compare above result to the correlation coefficient we obtained using the Seaborn library in Chapter 1.3. 

For cosine similarity, we use the original (non-centered) data:

In [29]:
cossim=np.dot(x,y)/(la.norm(x)*la.norm(y))
print(f'Cosine similarity = {cossim :.3f}')

Cosine similarity = 0.991


---
**<font color="red">Example 2</font>**

Here, we will use simple arrays to show cosine similarity can be quite different from the correlation coefficient. In Example 1, cosine similarity was larger than the correlation coefficient. In this example, we will see the opposite.

In [31]:
x=np.array([1,2,3])
y=np.array([101, 102, 103])

xn=(x-x.mean())
yn=(y-y.mean())

In [32]:
corr=np.dot(xn,yn)/(la.norm(xn)*la.norm(yn))
print(f'Correlation coefficient = {corr:.3f}')

cossim=np.dot(x,y)/(la.norm(x)*la.norm(y))
print(f'Cosine similarity = {cossim :.3f}')

Correlation coefficient = 1.000
Cosine similarity = 0.929


######
<font color="#3f7fff" size="5"><div style="
    font-family: Verdana;
    font-size: 1.6em;
    font-weight: 600;
    line-height: 1.3;
    padding: 0.5em 1em;
    margin: 0.5em 0 0.6em 0;
    border-top: 2px solid rgba(144,62,80,0.3);
    border-bottom: 2px solid rgba(144,62,80,0.3);
    border-radius: 8px;
">
<b>Closed Leontief Input-Output Model</b>
</div></font> 


The **closed Leontief input-output model** describes an economic system consisting of several sectors, where each sector requires outputs from other sectors and there is no external demand. All production in the system is consumed within the system itself. The goal of the analysis is to find an equilibrium of the model; that is, production levels that balance the entire economy so that every sector receives exactly the inputs it needs, with no shortages and no overproduction. The economist Wassily Leontief showed that such systems always have an equilibrium solution.

---
**<font color="red">Example 3</font>**

Suppose our economy has three sectors: Transportation, Manufacturing and Energy. Each sector produces its corresponding product, and each sector consumes products from all sectors. The diagram and table below show how the products are distributed. For example, we can see that output of the Transportation sector is distributed as follows: 30% is consumed by the Transportation itself, 50% consumed by the Manufacturing and 20% by the Energy sector.

<center> <img src="data:image/png;base64,  ...
 " width="80%"> </center>

Because this is a closed model, all output must be used within the system. Therefore, the percentages in each column of the table must add up to $1$. 

We measure the total output of each sector by its monetary value, which we call its **price**. Let $${\bf p}=\begin{bmatrix} t \\ m \\ e \end{bmatrix}$$ be the production vector consisting of the prices of Transportation, Manufacturing and Energy sectors, denoted by $t$, $m$ and $e$ respectively. For the system to be in equilibrium, the value of the output produced by each sector must equal the value of the inputs it consumes. For example, the Transportation sector is consuming 30% of its own output, 40% of Manufacturing output and 30% of Energy output. Therefore the total value of inputs used by Transportation must equal its total production: $$t= 0.3t+0.4m+0.3e$$

Similarly, for other sectors we obtain

$$ m=0.5t+0.2m+0.5e$$
$$e=0.2t+0.4m+0.2e$$


Moving all terms to one side of each equation produces a homogeneous linear system:

$$ \left\{\begin{array}{l} \text{ }\text{ }0.7t&-0.4m&-0.3e&=0\\
-0.5t&+0.8m&-0.5e&=0\\
-0.2t&-0.4m&+0.8e&=0 \end{array}\right.$$


Before solving this system, let's reformulate it using matrices. We construct a **consumption matrix** $C$ that contains all input-output ratios $$C=\begin{bmatrix}  0.3  & 0.4 & 0.3  \\ 0.5 & 0.2  & 0.5 \\  0.2 & 0.4 & 0.2\end{bmatrix}$$ If ${\bf p}= \begin{bmatrix} t \\ m \\ e \end{bmatrix}$ is the vector of production levels, then 

$$C{\bf p}=\begin{bmatrix} 0.3t+0.4m+0.3e \\ 0.5t+0.2m+0.5e\\0.2t+0.4m+0.2e \end{bmatrix}$$ represents the **total value of consumption by each sector**. For equilibrium, production must equal consumption, so we must solve ${\bf p}=C{\bf p}$.
We can rewrite this equation as

$$I{\bf p}=C{\bf p}$$
$$I{\bf p}-C{\bf p}={\bf 0}$$
$$(I-C){\bf p}={\bf 0}$$

The matrix $I-C$ is called the **Leontief matrix**: $$I-C=\begin{bmatrix} 0.7  & -0.4 & -0.3 \\ -0.5 & 0.8  & -0.5   \\  -0.2 & -0.4 & 0.8\end{bmatrix}$$ 
Notice that this is exactly the matrix we would obtain from the homogeneous system above. The set of all equilibrium price vectors is therefore the null space of the Leontief matrix: $Nul(I-C)$. 

We compute the null space by row reducing the augmented matrix:



$$\left[
\begin{array}\fbox{\begin{array}{ccc|c}
0.7  & -0.4 & -0.3 & 0 \\ -0.5 & 0.8  & -0.5   & 0 \\  -0.2 & -0.4 & 0.8 & 0
\end{array}}\end{array}
\right]
{\sim} 
\cdots{\sim} 
\left[
\begin{array}\fbox{\begin{array}{ccc|c}
1  & 0  & -1.22 & 0 \\  0  & 1 & -1.39 & 0  \\  0 & 0 & 0 & 0
\end{array}}\end{array}
\right]
$$

We have one free variable $e$. Therefore, the equilibrium price vector is

$${\bf p}=\begin{bmatrix} t \\ m \\ e \end{bmatrix} = \begin{bmatrix} 1.22e \\ 1.39e \\ e \end{bmatrix}=e\begin{bmatrix} 1.22 \\ 1.39 \\ 1 \end{bmatrix}$$

Since the system has a free variable, there are infinitely many equilibrium price vectors. Any choice of $e$ would result in a valid equlibrium for our model. For instance, if $e=\$100$, then $t=\$122$ and $m=\$139$.


---
**<font color="red">Example 3 with Python</font>**

Let's find the equilibrium prices of all sectors if the production value of the Energy sector is $\$100$

We construct the consumption matrix $C$ and find the null space of $I-C$

In [21]:
C=np.array([[0.3, 0.4, 0.3],[0.5, 0.2, 0.5],[0.2, 0.4, 0.2]])
nullsp=la.null_space(np.eye(3)-C)
print(nullsp)

[[0.58]
 [0.66]
 [0.48]]


The price of the Energy sector is the third entry of the vector. We need to scale the vector so that its third entry is equal to $100$. We can do this by dividing the vector by its third entry and muliplying by $100$:

In [22]:
print(nullsp*100/nullsp[2])

[[122.22]
 [138.89]
 [100.  ]]


Therefore, if $e=\$100$, then $t  \approx \$122$ and $m \approx\$139$.

######
<font color="#3f7fff" size="5"><div style="
    font-family: Verdana;
    font-size: 1.6em;
    font-weight: 600;
    line-height: 1.3;
    padding: 0.5em 1em;
    margin: 0.5em 0 0.6em 0;
    border-top: 2px solid rgba(144,62,80,0.3);
    border-bottom: 2px solid rgba(144,62,80,0.3);
    border-radius: 8px;
">
<b>Open Leontief Input-Output Model</b>
</div></font> 


In the open model, each sector must not only supply inputs to other sectors but also produce goods for consumers outside the system (the general public). Therefore, there is an external demand that each sector must satisfy.

---
**<font color="red">Example 4</font>**

Suppose our economy has only two sectors: Energy and Manufacturing. The sectors consume part of each other's production as follows:
* Energy consumes 10% of its own production and 20% of Manufacturing production.
* Manufacturing consumes 40% of its own production and 30% of Energy production.

The remaining production is sold to consumers outside the system. Suppose the external demand requires $\$40$ million worth of Energy and $\$30$ worth million of Manufacturing. We want to determine the production levels required for equilibrium.

**Solution.**

We can construct the **consumption matrix** $C$ and the **demand vector** ${\bf d}$:

$$ C= \begin{bmatrix} 0.1 & 0.2 \\ 0.3 & 0.4 \end{bmatrix}, {\bf d}=\begin{bmatrix} 40\\30\end{bmatrix}$$


Let ${\bf p}$ represent the production vector. For equilibrium, total production must equal internal consumption plus external demand, so $${\bf p}=C{\bf p}+{\bf d}$$

Rearranging the equation as we did in the closed model gives $$(I-C){\bf p}={\bf d}$$
Unlike the closed model, this is a non-homogeneous linear system. 

Let's construct the augmented matrix for it and solve it.

$$\left[
\begin{array}\fbox{\begin{array}{cc|c}
0.9  & -0.2 & 40 \\ -0.3 & 0.6  & 30
\end{array}}\end{array}
\right]
{\sim} 
\cdots{\sim} 
\left[
\begin{array}\fbox{\begin{array}{cc|c}
1  & 0  & 62.5 \\  0  & 1 & 81.25 
\end{array}}\end{array}
\right]
$$

Therefore, the equilibrium production levels are:
* Energy should produce $\$62.5$ million 
* Manufacturing should produce $\$81.25$ million

This means the sectors must produce these amounts in order to satisfy both internal consumption and external demand.

<div style="
    font-family: Verdana, sans-serif;
    font-size: 14px;
    line-height: 1.6;
    margin: 0.8em 0;
    padding: 0.75em 1em;
    border-radius: 6px;
    border-left: 4px solid #2c3e50;
    border-left-color: #4c72b0;
    background-color: rgba(10,150,250,0.16);
    margin: 0.4em 0;
">
<strong>📌 Note:</strong> Unlike the closed model, an open Leontief model does not always have a solution. Depending on the consumption structure and external demand, the resulting production levels may be negative or otherwise economically unrealistic.
</div>


---
**<font color="red">Example 4 with Python</font>**

We can also solve this using Python

In [23]:
C=sp.Matrix([[0.1, 0.2],[0.3,0.4]])
d=sp.Matrix([40, 30])

sp.linsolve((sp.eye(2)-C,d))

{(62.5, 81.25)}

Notice that, since we have a unique solution, we can also use the NumPy or SciPy solver:

In [24]:
C=np.array([[0.1, 0.2],[0.3,0.4]])
d=np.array([40, 30])

p=la.solve(np.eye(2)-C,d)
print(p)

[62.5  81.25]


######
<font color="#3f7fff" size="5"><div style="
    font-family: Verdana;
    font-size: 1.6em;
    font-weight: 600;
    line-height: 1.3;
    padding: 0.5em 1em;
    margin: 0.5em 0 0.6em 0;
    border-top: 2px solid rgba(144,62,80,0.3);
    border-bottom: 2px solid rgba(144,62,80,0.3);
    border-radius: 8px;
">
<b>Network Flow</b>
</div></font> 

Systems of linear equations also arise in the study of **network flows**. Network flow models appear in many real-world applications, such as traffic flow in cities, electric current flow in circuits, data traffic on the internet, supply chains and many more. A **network** consists of a set of points called **nodes**, connected by directed lines called **branches**. Each branch carries a certain flow rate, which may be known or unknown. The main assumption in network flow models is that  flow is conserved. This means that at each node, the total flow entering the node equals the total flow leaving the node. A typical question in network flow models is how to determine the flow rates along unknown branches when only partial information about the network is available.

---
**<font color="red">Example 5</font>**

The network in the following figure represents traffic flow along four one-way streets with four intersections (shown in blue). The numbers indicate the number of cars per hour traveling along each street. Some streets have unknown traffic flow (shown in red). Our goal is to determine how many cars pass through those streets.

<center> <img src="data:image/png;base64,  ...
 " width="40%"> </center>

Because traffic is conserved, the number of cars entering an intersection must equal the number leaving it. Therefore, we can write an equation for each intersection. 

For example, consider intersection $A$. Two streets bring traffic into the intersection, with flows $x_2$ and $x_3$, while two streets carry traffic away from the intersection, with flows $x_1$ and $80$. Therefore, for $A$ we have $$ x_2+x_3=80+x_1$$
Applying the same idea to the other intersections $B$, $C$ and $D$, we obtain:
$$ 100+ 60 = x_2+x_4$$
$$x_4+x_5=120$$
$$150+110=x_3+x_5$$

Rearranging the equations gives the non-homogeneous linear system:

$$\begin{align*}\left\{\begin{array}{l}
  -x_1 &+x_2&+x_3&& &=80 \\
   &x_2&&+x_4& &=160\\
   &&&x_4&+x_5&=120\\
   &&x_3&&+x_5&=260
\end{array}\right.
\qquad \Rightarrow \qquad 
\left[
\begin{array}{ccccc|c}
-1 & 1  & 1 &0&0&80\\ 0 & 1  & 0&1&0&160 \\0&0&0&1&1&120\\0&0&1&0&1&260 
\end{array}
\right]\end{align*}
$$

Solving the system gives

$$\left\{\begin{array}{l}
  x_1=220 \\
  x_2=40+x_5\\
  x_3=260-x_5\\
  x_4=120-x_5\\
  x_5=free
\end{array}\right.$$

Because there is a free variable, the system has infinitely many solutions. However, since traffic flows cannot be negative, we must impose some additional restrictions. For example, $x_4=120-x_5$ must be nonnegative, so $x_5 \leq 120 $. Based on this, we can safely say that traffic along $x_2$ is between $40$ and $160$ cars per hour, and traffic along $x_3$ is between $140$ and $260$ cars per hour.

---
**<font color="red">Example 5 with Python</font>**

First, we will use SymPy


In [40]:
A=sp.Matrix([[-1, 1, 1, 0, 0],[0,1,0,1,0],[0,0,0,1,1],[0,0,1,0,1]])
b=sp.Matrix([80, 160, 120, 260])
sp.linsolve((A, b))

{(220, tau0 + 40, 260 - tau0, 120 - tau0, tau0)}

This looks exactly like the solution we obtained above.

Next, we will use NumPy.

In [41]:
A=np.array([[-1, 1, 1, 0, 0],[0,1,0,1,0],[0,0,0,1,1],[0,0,1,0,1]])
b=np.array([80, 160, 120, 260])
x=la.null_space(A).round(3)
print(x)

[[ 0. ]
 [-0.5]
 [ 0.5]
 [ 0.5]
 [-0.5]]


In [42]:
x=la.lstsq(A,b)[0]
print(x)

[220. 125. 175.  35.  85.]


This produces the following solution 
$$\left\{\begin{array}{l}
  x_1=220 \\
  x_2=125-0.5t\\
  x_3=175+0.5t\\
  x_4=35+0.5t\\
  x_5=85-0.5t
\end{array}\right.$$

It may look different, but it describes exactly the same solution as before. For instance, notice that $t$ must be at least $-70$, otherwise $x_4$ would be negative. If $t \geq -70$, then $x_5 \leq 120$, which is exactly the restriction we obtained before.

######
<font color="#0fbf0f" size="5"><div style="
    font-family: Verdana;
    font-size: 1.6em;
    font-weight: 600;
    line-height: 1.3;
    padding: 0.5em 1em;
    margin: 0.5em 0 0.6em 0;
    border-top: 2px solid rgba(144,62,80,0.3);
    border-bottom: 2px solid rgba(144,62,80,0.3);
    border-radius: 8px;
">
<b>Exercises</b>
</div></font> 

**1.** 
> Let ${\bf v}=\begin{bmatrix} 1 \\ 1 \\ 0 \end{bmatrix}$ and ${\bf u}=\begin{bmatrix} 1 \\ 0 \\ 1 \end{bmatrix}$. Compute without using Python the correlation coefficient and cosine similarity.

**2.**  
> Find the correlation coefficient and cosine similarity between Math and Computer Science in "Grades Set" using Python.


**3.**  
> Find the correlation coefficient and cosine similarity between Math and Art in "Grades Set" using Python.




**4.**  
> Consider a closed economy with two sectors: Energy (E) and Manufacturing (M). Of the Energy sector's output, 40% goes to Energy and 60% goes to Manufacturing
Of the Manufacturing sector's output, 30% goes to Energy and 70% goes to Manufacturing. <br><br>
>***a.*** Draw a diagram representing this system. <br>
>***b.*** Find the consumption matrix and the Leontief matrix. <br>
>***c.*** Find equlibrium prices if Energy sector's output is $\$1000$. <br>

<center> <img src="data:image/png;base64,  ...
 " width="30%"> </center>

**5.**  
> A streaming platform ecosystem has three sectors:<br>
> * Content Creation (C) – studios producing movies and shows;<br>
> * Platform Technology (T) – servers, streaming infrastructure;<br>
> * Marketing & Promotion (M) – advertising and subscriber acquisition.
> 
>All revenue circulates inside the platform ecosystem, so the system is closed. The distribution of each sector's output is given by the table below.<br>
>
>***a.*** Draw a diagram representing this system<br>
>***b.*** Find the consumption matrix and the Leontief matrix<br>
>***c.*** Find the equilibrium revenue of each sector if Content Creation should produce $\$1000$ of revenue.<br>
>***d.*** Find the equilibrium revenue of each sector if the total revenue consumed by the Content Creation sector is $\$1000$ (Be careful, this question is different from c)<br>
>***e.*** Find the equilibrium revenue of each sector if the total ecosystem is worth $\$1000$. (This means the sum $c+t+m=1000$)<br>

<center> <img src="data:image/png;base64,  ...
 " width="30%"> </center>

**6.** 
> In a fantasy video game, a guild of alchemists produces two magical items: Healing Potions (H) and Mana Potion (M). Creating these items requires ingredients obtained from previously produced potions. Each Healing Potion requires 0.3 units of Healing Potions and 0.2 units of Mana Potion. Each Mana Potion requires 0.4 units of Healing Potion and 0.1 units of Mana Potion. Adventurers visiting the guild create external demand for potions: 6 Healing Potion and 4 Mana Potion. <br><br>
> ***a.*** Set up consumption matrix $C$ and demand vector ${\bf d}$.<br>
> ***b.*** How many of each potion the guild should produce to satisfy the demand?



**7.**  
> Traffic flow around a city block is given in the diagram below.<br><br>
> ***a.*** Set up and solve a linear system to find the unknowns.<br>
> ***b.*** What is the minimum traffic flow along $x_4$? What is the minimum traffic flow along $x_1$?

<center> <img src="data:image/png;base64,  ...
 " width="30%"> </center>

**8.** 
> A water flow in the pipe network is given below.<br><br>
> ***a.*** Set up and solve the system to find all the unknowns.<br>
> ***b.*** If we close pipes $x_6$ and $x_7$, what are the flows in other pipes?<br>
> ***c.*** If we close pipes $x_1$ and $x_2$, what happens in pipes $x_6$ and $x_7$?

<center> <img src="data:image/png;base64,  ...
 " width="40%"> </center>